# 🔍 Système Juridique Intelligent Marocain
## Notebook 03 — Pipeline RAG (Retrieval-Augmented Generation)

---

Ce notebook couvre :
1. Chargement des embeddings et de l'index FAISS
2. Recherche sémantique
3. Construction du contexte RAG
4. Intégration avec Claude (Anthropic API)
5. Tests d'inférence
6. Évaluation de la qualité du retrieval

In [9]:
# ============================================================
#  CELLULE 1 — Installation
# ============================================================
import subprocess, sys
pkgs = ['sentence-transformers','faiss-cpu','anthropic','pandas','numpy',
        'matplotlib','seaborn']
for p in pkgs:
    subprocess.run([sys.executable,'-m','pip','install',p,'-q',
                    '--break-system-packages'], capture_output=True)
print('✅ Prêt')

✅ Prêt


In [10]:
# ============================================================
#  CELLULE 2 — Imports
# ============================================================
import os, pickle, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

VEC_DIR  = '../vector_db'
PROC_DIR = '../processed_data'
print('✅ Imports OK')

✅ Imports OK


---
## 1. Chargement de la base vectorielle

In [11]:
# ============================================================
#  CELLULE 3 — Chargement FAISS + metadata
# ============================================================
import faiss

index    = faiss.read_index(os.path.join(VEC_DIR, 'faiss_index.bin'))
with open(os.path.join(VEC_DIR, 'metadata.pkl'), 'rb') as f:
    metadata = pickle.load(f)

embeddings = np.load(os.path.join(VEC_DIR, 'embeddings.npy'))

print(f'✅ Index FAISS chargé : {index.ntotal} vecteurs')
print(f'   Metadata          : {len(metadata)} articles')
print(f'   Embeddings shape  : {embeddings.shape}')

✅ Index FAISS chargé : 45 vecteurs
   Metadata          : 45 articles
   Embeddings shape  : (45, 384)


In [12]:
# ============================================================
#  CELLULE 4 — Chargement du modèle d'embedding
# ============================================================
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
embed_model = SentenceTransformer(MODEL_NAME)
print(f'✅ Modèle d\'embedding : {MODEL_NAME}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Modèle d'embedding : paraphrase-multilingual-MiniLM-L12-v2


---
## 2. Moteur de recherche sémantique

In [13]:
# ============================================================
#  CELLULE 5 — Classe RAGRetriever
# ============================================================
class MoroccanLegalRetriever:
    """Moteur de recherche sémantique sur les lois marocaines.
    
    Utilise FAISS pour la recherche par similarité cosinus
    sur des embeddings multilingues (FR / AR).
    """

    def __init__(self, faiss_index, metadata: list, embed_model):
        self.index      = faiss_index
        self.metadata   = metadata
        self.model      = embed_model

    def search(self, query: str, top_k: int = 5,
               category_filter: str = None) -> list:
        """Recherche les articles les plus pertinents.
        
        Args:
            query           : Question ou situation de l'utilisateur
            top_k           : Nombre de résultats à retourner
            category_filter : Filtrer par catégorie ('civil','penal',...)
        
        Returns:
            Liste de dicts {score, rank, ...metadata}
        """
        # 1. Encoder la requête
        q_emb = self.model.encode([query], normalize_embeddings=True)
        q_emb = q_emb.astype(np.float32)

        # 2. Recherche FAISS
        distances, indices = self.index.search(q_emb, top_k * 3)

        # 3. Construire les résultats
        results = []
        seen_texts = set()
        for dist, idx in zip(distances[0], indices[0]):
            if idx < 0 or idx >= len(self.metadata):
                continue
            meta = self.metadata[idx].copy()

            # Filtre catégorie
            if category_filter and meta.get('category') != category_filter:
                continue

            # Dédoublonnage par texte
            text_key = meta.get('article_text','')[:100]
            if text_key in seen_texts:
                continue
            seen_texts.add(text_key)

            meta['score'] = float(dist)
            meta['rank']  = len(results) + 1
            results.append(meta)

            if len(results) >= top_k:
                break

        return results

    def format_context(self, results: list) -> str:
        """Formate les résultats en contexte pour le LLM."""
        if not results:
            return 'Aucun article de loi pertinent trouvé.'

        ctx = ''
        for r in results:
            ctx += f"""\n[{r['rank']}] {r.get('title','Loi marocaine')} """
            ctx += f"""— Art. {r.get('article_number','?')} """
            ctx += f"""(Score: {r['score']:.3f})\n"""
            ctx += f"""Catégorie : {r.get('category','?')}\n"""
            ctx += f"""Texte : {r.get('article_text','')}\n"""
            ctx += '-' * 60 + '\n'
        return ctx


retriever = MoroccanLegalRetriever(index, metadata, embed_model)
print('✅ MoroccanLegalRetriever initialisé')

✅ MoroccanLegalRetriever initialisé


---
## 3. Intégration avec le LLM (Claude)

In [14]:
# ============================================================
#  CELLULE 6 — Classe RAGAgent (LLM + Retrieval)
# ============================================================
import anthropic

SYSTEM_PROMPT_FR = """Vous êtes un assistant juridique expert en droit marocain.
Votre rôle est d'analyser la situation décrite par l'utilisateur et de fournir
des conseils juridiques basés EXCLUSIVEMENT sur les articles de loi fournis.

INSTRUCTIONS :
1. Analysez attentivement la situation de l'utilisateur
2. Identifiez les lois applicables parmi les articles fournis
3. Expliquez clairement les droits et obligations de l'utilisateur
4. Indiquez les procédures recommandées
5. Citez précisément les articles de loi utilisés (numéro et titre)
6. Restez objectif et ne donnez pas d'avis personnel
7. Si la situation dépasse vos informations disponibles, conseillez de consulter un avocat

FORMAT DE RÉPONSE :
## 📋 Analyse de la situation
[Résumé de la situation]

## ⚖️ Lois applicables
[Articles pertinents avec citations]

## 🛡️ Droits et obligations
[Droits et obligations de l'utilisateur]

## 📝 Procédure recommandée
[Étapes à suivre]

## 📚 Références
[Liste des articles cités]"""

SYSTEM_PROMPT_AR = """أنت مساعد قانوني متخصص في القانون المغربي.
مهمتك تحليل الوضع الموصوف من قبل المستخدم وتقديم مشورة قانونية بناءً EXCLUSIVEMENT على مقالات القانون المقدمة.

التعليمات:
1. حلل وضع المستخدم بعناية
2. حدد القوانين المنطبقة من بين المقالات المقدمة
3. اشرح بوضوح حقوق والتزامات المستخدم
4. أشر إلى الإجراءات الموصى بها
5. اقتبس بدقة مواد القانون المستخدمة (رقم وعنوان)
6. كن موضوعياً ولا تبدِ رأياً شخصياً
7. إذا تجاوز الوضع المعلومات المتاحة لك، انصح باستشارة محامٍ"""


class MoroccanLegalAgent:
    """Agent IA juridique marocain.
    Combine la recherche sémantique (FAISS) et la génération (Claude).
    """

    def __init__(self, retriever: MoroccanLegalRetriever, api_key: str = None):
        self.retriever = retriever
        # Utiliser la clé depuis l'env si non fournie
        self.client    = anthropic.Anthropic(
            api_key=api_key or os.environ.get('ANTHROPIC_API_KEY', '')
        )

    def answer(self, question: str, lang: str = 'fr',
               top_k: int = 5, category: str = None) -> dict:
        """Répond à une question juridique.
        
        Returns:
            dict avec keys: response, retrieved_articles, query_time
        """
        t0 = time.time()

        # 1. Retrieval
        articles = self.retriever.search(question, top_k=top_k,
                                         category_filter=category)
        context  = self.retriever.format_context(articles)

        # 2. Construction du prompt
        system = SYSTEM_PROMPT_AR if lang == 'ar' else SYSTEM_PROMPT_FR
        user_prompt = f"""SITUATION/QUESTION DE L'UTILISATEUR :
{question}

ARTICLES DE LOI PERTINENTS (base de données juridique marocaine) :
{context}

Veuillez analyser cette situation et fournir une réponse juridique complète."""

        # 3. Génération LLM
        message = self.client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=1500,
            system=system,
            messages=[{'role': 'user', 'content': user_prompt}]
        )
        response = message.content[0].text
        query_time = time.time() - t0

        return {
            'response'          : response,
            'retrieved_articles': articles,
            'context_used'      : context,
            'query_time_s'      : round(query_time, 2)
        }


print('✅ MoroccanLegalAgent défini')
print('   Utilisation : agent.answer(question, lang="fr", top_k=5)')

✅ MoroccanLegalAgent défini
   Utilisation : agent.answer(question, lang="fr", top_k=5)


---
## 4. Tests d'inférence

In [15]:
# ============================================================
#  CELLULE 7 — Tests de recherche sémantique (sans LLM)
# ============================================================
TEST_QUERIES = [
    ('Mon employeur veut me licencier sans raison valable.', 'fr', 'social'),
    ('J\'ai été victime de violences physiques de la part de mon conjoint.', 'fr', 'penal'),
    ('Je souhaite acheter un appartement, quelles sont les règles du contrat de vente ?', 'fr', 'civil'),
    ('Mon enfant a été exclu de l\'université après avoir triché à un examen.', 'fr', 'education'),
    ('أريد معرفة حقوقي كعامل في حالة الفصل من العمل', 'ar', None),
]

print('🔍 TESTS DE RECHERCHE SÉMANTIQUE\n')
print('=' * 70)

retrieval_results = []
for query, lang, cat in TEST_QUERIES:
    print(f'\n📌 Query [{lang}] : {query[:60]}...')
    results = retriever.search(query, top_k=3, category_filter=cat)
    for r in results:
        print(f'   [{r["rank"]}] Score={r["score"]:.3f} | '
              f'{r.get("title","?")[:40]} Art.{r.get("article_number","?")} '
              f'[{r.get("category","?")}]')
    retrieval_results.append({'query': query, 'lang': lang, 'results': results})

print('\n✅ Tests de recherche terminés')

🔍 TESTS DE RECHERCHE SÉMANTIQUE


📌 Query [fr] : Mon employeur veut me licencier sans raison valable....
   [1] Score=0.551 | Code du Travail Art.39 [social]
   [2] Score=0.541 | Code du Travail Art.35 [social]
   [3] Score=0.533 | Conditions de travail des femmes Art.17 [social]

📌 Query [fr] : J'ai été victime de violences physiques de la part de mon co...
   [1] Score=0.497 | Loi relative à la lutte contre les viole Art.3 [penal]
   [2] Score=0.475 | Loi relative à la lutte contre les viole Art.2 [penal]
   [3] Score=0.404 | Loi relative à la lutte contre les viole Art.1 [penal]

📌 Query [fr] : Je souhaite acheter un appartement, quelles sont les règles ...
   [1] Score=0.593 | Code des Obligations et des Contrats Art.478 [civil]
   [2] Score=0.398 | Code des Obligations et des Contrats Art.19 [civil]
   [3] Score=0.395 | Code des Obligations et des Contrats Art.1 [civil]

📌 Query [fr] : Mon enfant a été exclu de l'université après avoir triché à ...
   [1] Score=0.415 | Loi relativ

In [16]:
# ============================================================
#  CELLULE 8 — Test complet RAG (nécessite clé API)
# ============================================================
# ⚠️  Définir ANTHROPIC_API_KEY dans l'environnement ou passer api_key=
API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'sk-ant-...')

# Instancier l'agent
agent = MoroccanLegalAgent(retriever, api_key=API_KEY)

# Test sur la première query
test_q = TEST_QUERIES[0]
print(f'🤖 TEST RAG COMPLET\n')
print(f'Question : {test_q[0]}\n')
print('-' * 60)

result = agent.answer(test_q[0], lang=test_q[1])

print(result['response'])
print(f'\n⏱️  Temps de réponse : {result["query_time_s"]}s')
print(f'📚 Articles utilisés : {len(result["retrieved_articles"])}')

🤖 TEST RAG COMPLET

Question : Mon employeur veut me licencier sans raison valable.

------------------------------------------------------------


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CcBcHtFFELkHSuRdv6yK7'}

In [1]:
# ============================================================
#  CELLULE 9 — Évaluation de la qualité du retrieval
# ============================================================
# Dataset d'évaluation : paires (question, catégorie_attendue)
eval_dataset = [
    ('licenciement abusif employeur', 'social'),
    ('violence conjugale femme', 'penal'),
    ('contrat de vente immobilier', 'civil'),
    ('divorce garde enfants', 'familial'),
    ('fraude examen scolaire', 'education'),
    ('amende produit alimentaire frauduleux', 'penal'),
    ('nationalité marocaine naturalisation', 'familial'),
    ('arbitrage litige commercial', 'civil'),
    ('congé annuel payé salarié', 'social'),
    ('kafala enfant abandonné', 'familial'),
]

hits_at_1 = 0  # La top-1 résultat est dans la bonne catégorie
hits_at_3 = 0  # Au moins 1 des 3 top résultats est dans la bonne catégorie
mrr_scores = []

for query, expected_cat in eval_dataset:
    results = retriever.search(query, top_k=5)
    cats = [r.get('category','') for r in results]

    if cats and cats[0] == expected_cat:
        hits_at_1 += 1
    if expected_cat in cats[:3]:
        hits_at_3 += 1

    # MRR
    rank = next((i+1 for i, c in enumerate(cats) if c == expected_cat), 0)
    mrr_scores.append(1/rank if rank > 0 else 0)

n = len(eval_dataset)
print('📊 ÉVALUATION DU RETRIEVAL')
print(f'   Recall@1 : {hits_at_1/n:.3f}  ({hits_at_1}/{n})')
print(f'   Recall@3 : {hits_at_3/n:.3f}  ({hits_at_3}/{n})')
print(f'   MRR      : {np.mean(mrr_scores):.3f}')

NameError: name 'retriever' is not defined

In [2]:
# ============================================================
#  CELLULE 10 — Visualisation des scores de similarité
# ============================================================
# Visualiser la distribution des scores pour une requête type
sample_query = 'contrat de travail licenciement'
all_results  = retriever.search(sample_query, top_k=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score par article
labels = [f"Art.{r['article_number']}\n{r.get('category','')}" for r in all_results]
scores = [r['score'] for r in all_results]
colors = plt.cm.RdYlGn([s for s in scores])

axes[0].barh(labels[::-1], scores[::-1], color=colors[::-1])
axes[0].set_title(f'Scores de similarité\n"{sample_query}"', fontweight='bold')
axes[0].set_xlabel('Score (cosine similarity)')
axes[0].set_xlim(0, 1)

# Distribution des scores par catégorie pour toutes les requêtes
cat_score_data = {}
for q, _, _ in TEST_QUERIES:
    res = retriever.search(q, top_k=3)
    for r in res:
        cat = r.get('category','?')
        cat_score_data.setdefault(cat, []).append(r['score'])

cats_plot = list(cat_score_data.keys())
data_plot = [cat_score_data[c] for c in cats_plot]
axes[1].boxplot(data_plot, labels=cats_plot, patch_artist=True)
axes[1].set_title('Distribution des scores par catégorie', fontweight='bold')
axes[1].set_ylabel('Score de similarité')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../processed_data/fig_rag_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Pipeline RAG complet et évalué')

NameError: name 'retriever' is not defined